# 308 - Model-Level Transcriptome–Phenotype Integration

## Objective

Construct the modeling-ready cohort by integrating harmonized transcriptomic profiles with the selected pharmacological phenotype representation.

This notebook serves as the bridge between the transcriptomic layer and the pharmacogenomic layer, generating the final analysis cohort used in downstream program-discovery workflows.

## Inputs

### Harmonized cohort

- `302_integrated_modeling_cohort.csv`

Provides the unified modeling universe and lineage annotations generated during the cohort harmonization phase.

### Transcriptomic layer

- `303_expression_harmonized.parquet`

Provides harmonized gene-expression profiles for all eligible models in the integrated cohort.

### Pharmacological layer

- `307_model_level_phenotype.parquet`

Provides the model-level pharmacological phenotype representation from the pharmacogenomic layer.

## Outputs

- Model-level pharmacological phenotype table
- Integrated transcriptome–phenotype cohort
- Integration audit report
- Coverage summaries
- Modeling-ready matrices

## Key Questions

- How many models contain both transcriptomic and pharmacological information?
- What is the final overlap between the molecular and pharmacological layers?
- Is integration coverage balanced across lineages?
- Are there lineage-specific losses during integration?
- What is the final modeling universe available for downstream program-discovery analyses?

---

In [1]:
# =============================================================================
# Imports
# =============================================================================

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import Paths

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# =============================================================================
# Project paths
# =============================================================================

PROJECT_ROOT = Paths.root

METADATA_DIR = Paths.metadata
EXPRESSION_DIR = Paths.expression
PHARMACOLOGY_DIR = Paths.pharmacology

OUTPUTS_DIR = Paths.pharmacology

COHORT_PATH = METADATA_DIR / "302_integrated_modeling_cohort.csv"

EXPRESSION_PATH = EXPRESSION_DIR / "303_expression_harmonized.parquet"

PHARMACOLOGY_PATH = PHARMACOLOGY_DIR / "307_model_level_phenotype.parquet"

In [3]:
# =============================================================================
# Load inputs
# =============================================================================

cohort = pd.read_csv(COHORT_PATH)

expression = pd.read_parquet(EXPRESSION_PATH)

pharmacology = pd.read_parquet(PHARMACOLOGY_PATH)

print("=" * 20)
print("COHORT")
print("=" * 20)
print(cohort.shape)

print("\n" + "=" * 20)
print("EXPRESSION")
print("=" * 20)
print(expression.shape)

print("\n" + "=" * 20)
print("PHARMACOLOGY")
print("=" * 20)
print(pharmacology.shape)

COHORT
(713, 8)

EXPRESSION
(713, 19194)

PHARMACOLOGY
(713, 7)


## Cohort Consistency Assessment

Before constructing the integrated analysis cohort, we verify that all input layers contain the same model universe and that no model-level mismatches were introduced during previous harmonization steps.

Because transcriptomic and pharmacological analyses will be linked at the model level, identifier consistency is a mandatory prerequisite for all downstream analyses.

In [4]:
# =============================================================================
# ModelID consistency assessment
# =============================================================================

cohort_ids = set(cohort["ModelID"])

expression_ids = set(expression["ModelID"])

pharmacology_ids = set(pharmacology["ModelID"])

print(f"Cohort models:       {len(cohort_ids):,}")
print(f"Expression models:   {len(expression_ids):,}")
print(f"Pharmacology models: {len(pharmacology_ids):,}")

print("\nOverlap checks")
print("-" * 40)

print(
    "Cohort == Expression:",
    cohort_ids == expression_ids
)

print(
    "Cohort == Pharmacology:",
    cohort_ids == pharmacology_ids
)

print(
    "Expression == Pharmacology:",
    expression_ids == pharmacology_ids
)

Cohort models:       713
Expression models:   713
Pharmacology models: 713

Overlap checks
----------------------------------------
Cohort == Expression: True
Cohort == Pharmacology: True
Expression == Pharmacology: True


In [5]:
print("\nColumns of cohort:")
print(cohort.columns.tolist())

print("\nFirst 20 columns of expression:")
print(expression.columns[:20].tolist())

print("\nColumns of pharmacology:")
print(pharmacology.columns.tolist())


Columns of cohort:
['ModelID', 'SangerModelID', 'COSMICID', 'CellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'CCLEName']

First 20 columns of expression:
['ModelID', 'TSPAN6 (7105)', 'TNMD (64102)', 'DPM1 (8813)', 'SCYL3 (57147)', 'C1orf112 (55732)', 'FGR (2268)', 'CFH (3075)', 'FUCA2 (2519)', 'GCLC (2729)', 'NFYA (4800)', 'STPG1 (90529)', 'NIPAL3 (57185)', 'LAS1L (81887)', 'ENPP4 (22875)', 'SEMA3F (6405)', 'CFTR (1080)', 'ANKIB1 (54467)', 'CYP51A1 (1595)', 'KRIT1 (889)']

Columns of pharmacology:
['ModelID', 'mean_ln_ic50', 'selected_phenotype', 'mean_auc', 'median_auc', 'mean_z_score', 'median_z_score']


## Phenotype Construction

Generate the final model-level pharmacological phenotype table using the selected primary phenotype representation (raw LN_IC50).

In [6]:
# =============================================================================
# Model-level phenotype table
# =============================================================================

phenotype = pharmacology[
    [
        "ModelID",
        "selected_phenotype"
    ]
].copy()

print(phenotype.shape)

display(phenotype.head())

(713, 2)


,ModelID,selected_phenotype
0,ACH-000001,3.836449
1,ACH-000002,2.296450
2,ACH-000004,2.460608
3,ACH-000006,1.338897
4,ACH-000007,3.013501


In [7]:
# =============================================================================
# Construct integrated analysis cohort
# =============================================================================

analysis_cohort = (
    cohort
    .merge(
        phenotype,
        on="ModelID",
        how="inner"
    )
)

print(analysis_cohort.shape)

display(
    analysis_cohort.head()
)

(713, 9)


,ModelID,SangerModelID,COSMICID,CellLineName,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,CCLEName,selected_phenotype
0,ACH-000001,SIDM00105,905933.0,NIH:OVCAR-3,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,NIHOVCAR3_OVARY,3.836449
1,ACH-000002,SIDM00829,905938.0,HL-60,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,2.296450
2,ACH-000004,SIDM00594,907053.0,HEL,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,2.460608
3,ACH-000006,SIDM01023,908148.0,MONO-MAC-6,Myeloid,Acute Myeloid Leukemia,Acute Monoblastic/Monocytic Leukemia,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.338897
4,ACH-000007,SIDM00677,907795.0,LS513,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,LS513_LARGE_INTESTINE,3.013501


In [8]:
# =============================================================================
# Integrate transcriptomic layer
# =============================================================================

analysis_matrix = (
    expression
    .merge(
        phenotype,
        on="ModelID",
        how="inner"
    )
)

print(analysis_matrix.shape)

(713, 19195)


In [9]:
# =============================================================================
# Save outputs
# =============================================================================

analysis_cohort.to_csv(
    OUTPUTS_DIR / "308_analysis_cohort.csv",
    index=False
)

analysis_matrix.to_parquet(
    OUTPUTS_DIR / "308_transcriptome_phenotype_dataset.parquet",
    index=False
)

print("Saved:")
print(" - 308_analysis_cohort.csv")
print(" - 308_transcriptome_phenotype_dataset.parquet")

Saved:
 - 308_analysis_cohort.csv
 - 308_transcriptome_phenotype_dataset.parquet


In [10]:
# =============================================================================
# Output audit
# =============================================================================

print("Analysis cohort:")
print(analysis_cohort.shape)

print("\nIntegrated analysis matrix:")
print(analysis_matrix.shape)

Analysis cohort:
(713, 9)

Integrated analysis matrix:
(713, 19195)


### Summary

Transcriptomic and pharmacology-derived phenotype data were successfully integrated at the model level.

The final modeling cohort contains 713 harmonized cell lines with matched transcriptomic profiles, lineage annotations, and continuous pharmacological phenotype measurements.

Two modeling-ready artifacts were generated:

* `308_analysis_cohort.csv`
* `308_transcriptome_phenotype_dataset.parquet`

These datasets constitute the primary inputs for downstream feature-selection and transcriptomic program-discovery workflows.